In [17]:
# ============================================================
# Cell 1 — Imports
# Strategy E: Class-Aware Selective Mixing
#
# What is Strategy E?
#   Instead of treating all 10 disease classes the same way,
#   we route each class to the image domain it performs best
#   in — based on the per-class results from A, B, C, D.
#
#   Color-dominant classes  → trained on color images only
#   Texture-dominant classes → trained on segmented images
#   Mixed-domain classes    → trained on 50/50 mix
#
#   This is the original research contribution of this project.
# ============================================================

import os
import shutil
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Fix random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print('✅ All imports successful!')
print(f'   TensorFlow version : {tf.__version__}')
print(f'   GPU available      : {tf.config.list_physical_devices("GPU")}')
print(f'   Random seeds fixed : 42')

✅ All imports successful!
   TensorFlow version : 2.16.2
   GPU available      : []
   Random seeds fixed : 42


In [ ]:
# ============================================================
# Cell 2a — Load Validation-Derived Per-Class Accuracy
#
# Loads the per-class VAL accuracy exported by the new cells
# added to 01_evaluation_color.ipynb, 02_evaluation_segmented.ipynb,
# and 03_evaluation_mixed.ipynb (run those notebooks first, on
# this machine, so the JSON files below actually exist).
#
# This lets us compute a routing suggestion from validation
# data instead of the test set, avoiding the leak where the
# routing table itself was chosen using the same split later
# used to report Strategy E's final test accuracy.
# ============================================================

import json as _json
from pathlib import Path as _Path

_ROUTING_DIR = _Path(r'D:\Development\8th Sem Project\TomatoClassification\outputs\routing_analysis')

_val_data = {}
for _tag in ['A', 'B', 'C']:
    _p = _ROUTING_DIR / f'strategy_{_tag}_val_per_class.json'
    if _p.exists():
        with open(_p) as f:
            _val_data[_tag] = _json.load(f)['per_class_accuracy']
    else:
        print(f'⚠️  Missing {_p.name} — run the val-set cell in the matching evaluation notebook first.')

_DOMAIN_FOR_TAG = {'A': 'color', 'B': 'segmented', 'C': 'mixed'}
_suggested_routing = None

if len(_val_data) == 3:
    print('📋 Val-derived routing suggestion (argmax of A/B/C per class):\n')
    print(f"  {'Class':<40} {'A':>7} {'B':>7} {'C':>7}   Suggested")
    print('-' * 80)
    _suggested_routing = {}
    _all_classes = sorted(_val_data['A'].keys())
    for _cls in _all_classes:
        _scores = {t: _val_data[t].get(_cls, 0.0) for t in ['A', 'B', 'C']}
        _best_tag = max(_scores, key=_scores.get)
        _suggested_routing[_cls] = _DOMAIN_FOR_TAG[_best_tag]
        print(f"  {_cls:<40} {_scores['A']:>7.1%} {_scores['B']:>7.1%} {_scores['C']:>7.1%}   {_suggested_routing[_cls]} ({_best_tag})")
    print('\n➡️  Compare this to CLASS_ROUTING below (currently test-set derived).')
    print('    If a class disagrees, update CLASS_ROUTING to match the')
    print('    val-derived suggestion before training Strategy E, then rebuild')
    print('    dataset/processed_selective from the updated table.')
else:
    print('\n⚠️  Val-derived routing suggestion unavailable — need all 3 JSON files')
    print('    (see 01/02/03_evaluation*.ipynb — new val-eval cell in each).')


In [18]:
# ============================================================
# Cell 2 — Paths & Configuration
#
# Strategy E needs access to BOTH the color dataset AND the
# segmented dataset, because different classes will be loaded
# from different source folders.
#
# We also define the class routing table here — this is the
# core logic of Strategy E derived from A vs B per-class results.
# ============================================================

# Source dataset paths (already exist from Strategies A and B)
COLOR_PATH     = r'D:\Development\8th Sem Project\TomatoClassification\dataset\processed'
SEGMENTED_PATH = r'D:\Development\8th Sem Project\TomatoClassification\dataset\processed_segmented'
MIXED_PATH     = r'D:\Development\8th Sem Project\TomatoClassification\dataset\processed_mixed'

# Where Strategy E's assembled dataset will be saved
SELECTIVE_PATH = r'D:\Development\8th Sem Project\TomatoClassification\dataset\processed_selective'

MODEL_PATH  = r'D:\Development\8th Sem Project\TomatoClassification\models'
OUTPUT_PATH = r'D:\Development\8th Sem Project\TomatoClassification\outputs'

IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = 10
EPOCHS      = 20
STRATEGY    = 'E_selective_V3'

os.makedirs(MODEL_PATH,  exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ── CLASS ROUTING TABLE ──────────────────────────────────────
# Each class is mapped to the image domain it performs best in.
#
# ⚠️ LEGACY / TEST-SET DERIVED — the percentages in the comments
# below (A=99.3% vs B=78.7%, etc.) came from the TEST split
# evaluation in 01/02/03_evaluation*.ipynb. Using test-set
# results to choose this routing, then evaluating Strategy E on
# that same test set, is a data leak — cross-check this table
# against `_suggested_routing` (computed from val data in the
# cell above) before trusting Strategy E's final test accuracy.
#
#   'color'     → use images from COLOR_PATH
#   'segmented' → use images from SEGMENTED_PATH
#   'mixed'     → use images from MIXED_PATH  (50/50 already built)
#
# Folder names must exactly match the class folder names
# in your dataset (as shown by flow_from_directory class_indices).
# ─────────────────────────────────────────────────────────────
CLASS_ROUTING = {
    # Color-dominant: A was best — background color is the signal
    'Tomato_Bacterial_spot'                    : 'color',     # A=99.3% vs B=78.7% (+20.6%)
    'Tomato_Early_blight'                      : 'color',     # A=82.0% best; C collapsed to 62%
    'Tomato_Leaf_Mold'                         : 'color',     # A=92.3% vs B=79.0% (+13.3%)
    'Tomato_healthy'                           : 'color',     # D=98.7% peak but color-base drove it

    # Texture-dominant: B was best — shape/texture clearer without background
    'Tomato_Spider_mites_Two_spotted_spider_mite' : 'segmented', # B=89.3% vs A=77.3% (+12.0%)
    'Tomato_Septoria_leaf_spot'                : 'segmented', # B=90.7% vs A=85.3% (+5.4%)
    'Tomato_Yellow_Leaf_Curl_Virus'            : 'segmented', # B=96.7% consistently best

    # Mixed-domain: C or D was best — benefits from both modalities
    'Tomato_Late_blight'                       : 'mixed',     # C=94.7% best across all strategies
    'Tomato_Target_Spot'                       : 'mixed',     # C=87.3% best
    'Tomato_mosaic_virus'                      : 'mixed',     # D=96.4%, C=93.8% — both mixed benefit
}

print(f'✅ Configuration ready!')
print(f'   Strategy        : {STRATEGY}')
print(f'   Image size      : {IMAGE_SIZE}')
print(f'   Batch size      : {BATCH_SIZE}')
print(f'   Epochs          : {EPOCHS}')
print(f'   Classes routed  : {len(CLASS_ROUTING)}')
print(f'\n   Routing summary :')
for domain in ['color', 'segmented', 'mixed']:
    classes = [c for c, d in CLASS_ROUTING.items() if d == domain]
    print(f'     {domain:12s} → {len(classes)} classes')

✅ Configuration ready!
   Strategy        : E_selective_V3
   Image size      : (224, 224)
   Batch size      : 32
   Epochs          : 20
   Classes routed  : 10

   Routing summary :
     color        → 4 classes
     segmented    → 3 classes
     mixed        → 3 classes


In [19]:
import os
import shutil

# 1. Configuration
SPLITS = ['train', 'val', 'test']

# Ensure these paths point to your existing Strategy A, B, and C data roots
DOMAIN_PATHS = {
    'color'     : COLOR_PATH,      # Strategy A root
    'segmented' : SEGMENTED_PATH,  # Strategy B root
    'mixed'     : MIXED_PATH       # Strategy C root
}

# 2. Use the same routing table defined in Cell 2
CLASS_ROUTING = CLASS_ROUTING

print(f"🔄 Building Strategy E dataset at: {SELECTIVE_PATH}")

# 3. Simple Copy Logic
for split in SPLITS:
    print(f"\n📂 Processing {split} split...")
    
    for class_name, domain in CLASS_ROUTING.items():
        # Define source and destination
        source_dir = os.path.join(DOMAIN_PATHS[domain], split, class_name)
        dest_dir = os.path.join(SELECTIVE_PATH, split, class_name)
        
        # Create destination folder
        os.makedirs(dest_dir, exist_ok=True)
        
        # Only copy if the folder is empty (saves time)
        if not os.listdir(dest_dir):
            if os.path.exists(source_dir):
                files = os.listdir(source_dir)
                for f in files:
                    shutil.copy2(os.path.join(source_dir, f), os.path.join(dest_dir, f))
                print(f"   ✅ {class_name:<50} [From {domain}] - {len(files)} images")
            else:
                print(f"   ⚠️  FAILED: Source not found: {source_dir}")
        else:
            print(f"   ⏭️  Skipping {class_name} (Already exists)")

print("\n✅ Strategy E Dataset Assembly Complete!")

🔄 Building Strategy E dataset at: D:\Development\8th Sem Project\TomatoClassification\dataset\processed_selective

📂 Processing train split...
   ⚠️  FAILED: Source not found: D:\Development\8th Sem Project\TomatoClassification\dataset\processed/train/Tomato_Bacterial_spot
   ⚠️  FAILED: Source not found: D:\Development\8th Sem Project\TomatoClassification\dataset\processed/train/Tomato_Early_blight
   ⚠️  FAILED: Source not found: D:\Development\8th Sem Project\TomatoClassification\dataset\processed/train/Tomato_Leaf_Mold
   ⚠️  FAILED: Source not found: D:\Development\8th Sem Project\TomatoClassification\dataset\processed/train/Tomato_healthy
   ⚠️  FAILED: Source not found: D:\Development\8th Sem Project\TomatoClassification\dataset\processed_segmented/train/Tomato_Spider_mites_Two_spotted_spider_mite
   ⚠️  FAILED: Source not found: D:\Development\8th Sem Project\TomatoClassification\dataset\processed_segmented/train/Tomato_Septoria_leaf_spot
   ⚠️  FAILED: Source not found: D:\Dev

In [20]:
# ============================================================
# Cell 4 — Verify Class Folder Names
#
# IMPORTANT: Run this cell before training.
# It checks that the class folder names in CLASS_ROUTING
# exactly match the actual folder names on disk.
#
# If you see ❌ mismatches, update the keys in CLASS_ROUTING
# in Cell 2 to match the printed 'Found on disk' names,
# then re-run Cell 3 with force_rebuild=True.
# ============================================================

print('🔍 Verifying class folder names...\n')

# Check what folders actually exist in the color dataset train split
color_train = os.path.join(COLOR_PATH, 'train')
actual_folders = sorted([
    f for f in os.listdir(color_train)
    if os.path.isdir(os.path.join(color_train, f))
])

print('📋 Folders found on disk (Color train split):')
for f in actual_folders:
    print(f'   {f}')

print('\n📋 CLASS_ROUTING keys (what we defined):')
all_match = True
for cls_name, domain in CLASS_ROUTING.items():
    exists = cls_name in actual_folders
    mark = '✅' if exists else '❌ MISMATCH — update CLASS_ROUTING key'
    print(f'   {mark}  {cls_name}  →  {domain}')
    if not exists:
        all_match = False

print()
if all_match:
    print('✅ All class names verified — no mismatches found!')
    print('   Safe to proceed to Cell 5.')
else:
    print('❌ Mismatches found — fix CLASS_ROUTING keys in Cell 2')
    print('   then re-run Cell 3 with force_rebuild=True')

🔍 Verifying class folder names...



FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Development\\8th Sem Project\\TomatoClassification\\dataset\\processed/train'

In [ ]:
# ============================================================
# Cell 5 — Data Generators
#
# Same augmentation as Strategy A/B/C — consistent across
# all strategies so results are fairly comparable.
#
# The selective dataset folder structure is identical to
# the other datasets, so ImageDataGenerator works exactly
# the same way — it doesn't need to know about the routing.
# ============================================================

# Training — with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    horizontal_flip=True,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# Validation & Test — only rescale, no augmentation
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(SELECTIVE_PATH, 'train'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(SELECTIVE_PATH, 'val'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(SELECTIVE_PATH, 'test'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Show which domain each class ended up loading from
print('✅ Data generators ready!')
print(f'   🏋️  Training   : {train_generator.samples:,} images')
print(f'   ✔️  Validation : {val_generator.samples:,} images')
print(f'   🧪 Test       : {test_generator.samples:,} images')
print(f'\n   📋 Classes loaded ({len(train_generator.class_indices)}):')
for cls, idx in train_generator.class_indices.items():
    domain = CLASS_ROUTING.get(cls, 'unknown')
    domain_tag = {'color': '🔵 color', 'segmented': '🟠 segmented', 'mixed': '🟢 mixed'}.get(domain, '❓ unknown')
    print(f'      {idx}: {cls:<55} {domain_tag}')

In [ ]:
# ============================================================
# Cell 6 — Visualise the Routing Distribution
#
# A quick bar chart showing how many training images came
# from each domain. Useful for the paper to show the dataset
# composition is intentional and data-driven.
# ============================================================

train_dir = os.path.join(SELECTIVE_PATH, 'train')
class_counts   = {'color': [], 'segmented': [], 'mixed': []}
class_names_ordered = []

for cls, domain in CLASS_ROUTING.items():
    cls_dir = os.path.join(train_dir, cls)
    n = len(os.listdir(cls_dir)) if os.path.exists(cls_dir) else 0
    short_name = cls.replace('Tomato_', '').replace('_', ' ')
    class_names_ordered.append((short_name, domain, n))

# Sort for readability
domain_order = {'color': 0, 'segmented': 1, 'mixed': 2}
class_names_ordered.sort(key=lambda x: (domain_order[x[1]], x[0]))

labels  = [x[0] for x in class_names_ordered]
counts  = [x[2] for x in class_names_ordered]
domains = [x[1] for x in class_names_ordered]
colors  = {'color': '#2196F3', 'segmented': '#FF9800', 'mixed': '#4CAF50'}
bar_colors = [colors[d] for d in domains]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(labels, counts, color=bar_colors, edgecolor='white', linewidth=0.8)

# Value labels on bars
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(bar.get_height()), ha='center', va='bottom', fontsize=8)

# Legend
legend_patches = [
    mpatches.Patch(color='#2196F3', label='Color domain'),
    mpatches.Patch(color='#FF9800', label='Segmented domain'),
    mpatches.Patch(color='#4CAF50', label='Mixed domain'),
]
ax.legend(handles=legend_patches, fontsize=9)
ax.set_title('Strategy E — Training images per class by assigned domain',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Image count')
ax.set_xlabel('Disease class')
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, f'routing_distribution_{STRATEGY}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✅ Routing distribution chart saved!')
print(f'   Color domain     : {sum(1 for d in domains if d=="color")} classes')
print(f'   Segmented domain : {sum(1 for d in domains if d=="segmented")} classes')
print(f'   Mixed domain     : {sum(1 for d in domains if d=="mixed")} classes')

In [ ]:
# ============================================================
# Cell 7 — Build the Model
#
# Exact same architecture as Strategy A, B, C — fresh
# MobileNetV2 from ImageNet weights, frozen base, custom head.
#
# Why not start from Strategy A weights (like D did)?
#   Strategy E is a fresh training experiment — we want to
#   measure the effect of DATA COMPOSITION alone, not combine
#   it with fine-tuning effects from A. Clean comparison.
#
# If Strategy E beats Strategy A from a fresh start, that is
# a much stronger research claim than E+fine-tuning vs A.
# ============================================================

# Load MobileNetV2 pretrained on ImageNet, without its top layer
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base — preserve ImageNet features, only train the head
base_model.trainable = False

# Add the same custom classification head as A/B/C/D
x = base_model.output
x = GlobalAveragePooling2D()(x)       # reduce spatial dimensions to 1D
x = Dense(128, activation='relu')(x)  # learn task-specific patterns
x = Dropout(0.3)(x)                   # prevent overfitting
output = Dense(NUM_CLASSES, activation='softmax')(x)  # 10 class probabilities

model = Model(inputs=base_model.input, outputs=output)

# Same optimizer and lr as A/B/C — fair comparison
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable_params   = sum(tf.size(w).numpy() for w in model.trainable_weights)
untrainable_params = sum(tf.size(w).numpy() for w in model.non_trainable_weights)

print('✅ Model built successfully!')
print(f'   Architecture    : MobileNetV2 + GAP + Dense(128) + Dropout(0.3) + Softmax(10)')
print(f'   Total layers    : {len(model.layers)}')
print(f'   Trainable params: {trainable_params:,}   (custom head only)')
print(f'   Frozen params   : {untrainable_params:,}  (MobileNetV2 base)')
print(f'   Optimizer       : Adam  lr=0.001')
print(f'   Loss            : categorical_crossentropy')

In [ ]:
# ============================================================
# Cell 8 — Train the Model
#
# Identical callbacks to A/B/C:
#   EarlyStopping    — stops if val_loss doesn't improve for 5 epochs
#   ModelCheckpoint  — saves best val_accuracy model automatically
#   ReduceLROnPlateau — halves lr if val_loss stalls for 3 epochs
#
# Expected behaviour:
#   If Strategy E works, you should see val_accuracy stabilise
#   above 89.06% (Strategy A baseline) — that means the
#   class-aware routing improved on the best previous result.
# ============================================================

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(MODEL_PATH, f'best_model_{STRATEGY}.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print(f'🚀 Starting training — Strategy {STRATEGY}...')
print(f'   Dataset : Class-aware selective mix (4 color / 3 segmented / 3 mixed)')
print(f'   Baseline to beat : Strategy A = 89.06%\n')

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks
)

print(f'\n✅ Training complete — Strategy {STRATEGY}!')
print(f'   Best Val Accuracy : {max(history.history["val_accuracy"]):.2%}')
print(f'   Strategy A was    : 89.06%')
diff = max(history.history['val_accuracy']) - 0.8906
sign = '+' if diff >= 0 else ''
print(f'   E vs A (val)      : {sign}{diff*100:.2f}%')

In [ ]:
# ============================================================
# Cell 9 — Plot Training Curves
#
# Dashed reference lines show Strategy A's test results
# so you can instantly see whether E beats the best baseline.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],
             label='Train accuracy', color='#9C27B0', linewidth=2)
axes[0].plot(history.history['val_accuracy'],
             label='Val accuracy',   color='#E040FB', linewidth=2)
axes[0].axhline(y=0.8906, color='#2196F3', linewidth=1.5,
                linestyle='--', label='Strategy A baseline (89.06%)')
axes[0].set_title(f'Model Accuracy — Strategy {STRATEGY}',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.6, 1.0])

# Loss
axes[1].plot(history.history['loss'],
             label='Train loss', color='#F44336', linewidth=2)
axes[1].plot(history.history['val_loss'],
             label='Val loss',   color='#FF9800', linewidth=2)
axes[1].axhline(y=0.3366, color='#2196F3', linewidth=1.5,
                linestyle='--', label='Strategy A baseline (0.3366)')
axes[1].set_title(f'Model Loss — Strategy {STRATEGY}',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, f'training_curves_{STRATEGY}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✅ Training curves saved!')
print(f'   Best Train Accuracy : {max(history.history["accuracy"]):.2%}')
print(f'   Best Val Accuracy   : {max(history.history["val_accuracy"]):.2%}')
print(f'   Final Train Loss    : {history.history["loss"][-1]:.4f}')
print(f'   Final Val Loss      : {history.history["val_loss"][-1]:.4f}')
print(f'   Total epochs run    : {len(history.history["loss"])}')

In [ ]:
# ============================================================
# Cell 10 — Evaluate on Test Set
#
# Same evaluation as all previous strategies.
# We also run a full 5-strategy comparison here so you have
# a complete ranking A/B/C/D/E in one place.
# ============================================================

print(f'🔄 Evaluating Strategy {STRATEGY} on test set...\n')

test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

print(f'\n{"="*50}')
print(f'📊 Strategy {STRATEGY} — Test Results')
print(f'{"="*50}')
print(f'✅ Test Accuracy : {test_accuracy:.2%}')
print(f'📉 Test Loss     : {test_loss:.4f}')
print(f'{"="*50}')

# Full 5-strategy comparison
prev = [
    ('A', 'Color only',    0.8906),
    ('B', 'Segmented only',0.8613),
    ('C', 'Mixed 50/50',   0.8748),
    ('D', 'Fine-tuned A',  0.8842),
]

print(f'\n📈 All Strategies Comparison:')
all_accs = [(s, n, a) for s, n, a in prev] + [('E', 'Selective mix', test_accuracy)]
all_accs_sorted = sorted(all_accs, key=lambda x: x[2], reverse=True)
medals = ['🥇','🥈','🥉','  ','  ']
for i, (s, name, acc) in enumerate(all_accs_sorted):
    marker = ' ← new' if s == 'E' else ''
    print(f'  {medals[i]} Strategy {s} ({name:<18}) : {acc:.2%}{marker}')

print(f'\n   E vs A : {(test_accuracy - 0.8906)*100:+.2f}%')
print(f'   E vs B : {(test_accuracy - 0.8613)*100:+.2f}%')
print(f'   E vs C : {(test_accuracy - 0.8748)*100:+.2f}%')
print(f'   E vs D : {(test_accuracy - 0.8842)*100:+.2f}%')

print('\n✅ Test evaluation complete!')

In [ ]:
# ============================================================
# Cell 11 — Classification Report & Per-class Analysis
#
# This is the most important output for your paper.
# We compare Strategy E against all 4 previous strategies
# class-by-class to show exactly which classes improved
# and by how much — proving the routing logic worked.
# ============================================================

print(f'🔄 Generating per-class predictions...\n')

test_generator.reset()
y_pred_proba = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print(f'\n📋 Classification Report — Strategy {STRATEGY}\n')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

# ── Per-class comparison: all 5 strategies ──────────────────
# Previous results from evaluation notebooks
prev_class_acc = {
    'A': [99.3, 82.0, 89.3, 92.3, 85.3, 77.3, 86.0, 95.3, 91.1, 94.0],
    'B': [78.7, 74.7, 92.0, 79.0, 90.7, 89.3, 85.3, 96.7, 82.1, 90.0],
    'C': [90.0, 62.0, 94.7, 92.0, 89.3, 88.7, 87.3, 90.7, 93.8, 88.0],
    'D': [88.7, 74.7, 91.3, 92.3, 89.3, 85.3, 76.7, 96.0, 96.4, 98.7],
}

# Compute Strategy E per-class accuracy from confusion matrix
cm = confusion_matrix(y_true, y_pred)
e_acc = [cm[i, i] / cm[i].sum() * 100 for i in range(NUM_CLASSES)]

# Short display names (order must match test_generator.class_indices)
short_names = [
    c.replace('Tomato_', '').replace('_', ' ')[:22]
    for c in class_names
]

print('\n' + '─'*100)
print(f'{"Class":<24} {"A":>7} {"B":>7} {"C":>7} {"D":>7} {"E":>7}  {"E-A":>7}  {"Best"}')
print('─'*100)

e_wins, improvements = 0, []
for i, cls in enumerate(short_names):
    row = [prev_class_acc[s][i] for s in ['A','B','C','D']] + [e_acc[i]]
    best_val = max(row)
    best_strat = ['A','B','C','D','E'][row.index(best_val)]
    diff_a = e_acc[i] - prev_class_acc['A'][i]
    flag = '🟢' if diff_a > 0 else '🔴'
    e_mark = f'>>> {e_acc[i]:5.1f}%' if best_strat == 'E' else f'    {e_acc[i]:5.1f}%'
    print(f'{cls:<24} {prev_class_acc["A"][i]:>6.1f}% {prev_class_acc["B"][i]:>6.1f}% '
          f'{prev_class_acc["C"][i]:>6.1f}% {prev_class_acc["D"][i]:>6.1f}% '
          f'{e_mark}  {flag}{diff_a:>+5.1f}%  {best_strat}')
    if best_strat == 'E':
        e_wins += 1
    if diff_a > 0:
        improvements.append((cls, diff_a))

print('─'*100)
print(f'{"OVERALL":<24} {"89.06%":>7} {"86.13%":>7} {"87.48%":>7} {"88.42%":>7} {test_accuracy:>6.2%}')

print(f'\n✅ Per-class analysis complete!')
print(f'   Strategy E wins {e_wins}/10 classes outright')
print(f'   Improved vs Strategy A in {len(improvements)}/10 classes')
if improvements:
    improvements.sort(key=lambda x: x[1], reverse=True)
    print(f'   Biggest gain : {improvements[0][0]} ({improvements[0][1]:+.1f}% vs A)')

In [ ]:
# ============================================================
# Cell 12 — Confusion Matrix
# ============================================================

fig, ax = plt.subplots(figsize=(12, 10))

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(
    cm_norm,
    annot=True, fmt='.2f', cmap='Purples',
    xticklabels=short_names, yticklabels=short_names,
    ax=ax
)
ax.set_title(f'Confusion Matrix — Strategy {STRATEGY}',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.tick_params(axis='x', rotation=40)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, f'confusion_matrix_{STRATEGY}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('✅ Confusion matrix saved!')

In [ ]:
# ============================================================
# Cell 13 — Per-class Comparison Chart (All 5 Strategies)
#
# This is the main research chart for your paper.
# Five grouped bars per class — shows the full story of
# how Strategy E's class-aware routing affects each disease.
# ============================================================

x = np.arange(len(short_names))
width = 0.15   # narrower bars to fit 5 groups

fig, ax = plt.subplots(figsize=(18, 6))

offsets = [-2, -1, 0, 1, 2]
strategy_data = [
    ('A (Color)',     prev_class_acc['A'], '#2196F3', '///'),
    ('B (Segmented)', prev_class_acc['B'], '#FF9800', '\\\\\\'),
    ('C (Mixed)',     prev_class_acc['C'], '#4CAF50', '---'),
    ('D (Fine-tuned)',prev_class_acc['D'], '#9C27B0', '...'),
    ('E (Selective)', e_acc,               '#E91E63', 'xxx'),
]

for (label, data, color, hatch), offset in zip(strategy_data, offsets):
    bars = ax.bar(x + offset * width, data, width,
                  label=label, color=color, alpha=0.85,
                  hatch=hatch, edgecolor='white', linewidth=0.5)

# Reference line at Strategy A overall accuracy (baseline)
ax.axhline(y=89.06, color='#2196F3', linewidth=1.2,
           linestyle='--', alpha=0.6, label='Strategy A overall (89.06%)')

ax.set_xlabel('Disease Class', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Per-class Accuracy — All 5 Strategies (A/B/C/D/E)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=35, ha='right', fontsize=8.5)
ax.set_ylim([50, 110])
ax.legend(fontsize=8, loc='lower right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'comparison_all_strategies_ABCDE.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('✅ All-strategy comparison chart saved!')
print(f'   📁 {os.path.join(OUTPUT_PATH, "comparison_all_strategies_ABCDE.png")}')

In [ ]:
# ============================================================
# Cell 14 — Strategy E Routing Validation Chart
#
# This chart is unique to Strategy E — it shows for each class
# whether the routing decision was correct:
#   Did color classes actually score better with color images?
#   Did texture classes actually score better with segmented?
#   Did mixed classes actually score better with mixed?
#
# Green bar = E met or beat its assigned domain's best result
# Red bar   = E underperformed even the assigned domain
# This validates (or challenges) the routing logic empirically.
# ============================================================

# For each class: compare E's result against the assigned domain baseline
domain_baseline = {
    'color'     : prev_class_acc['A'],
    'segmented' : prev_class_acc['B'],
    'mixed'     : prev_class_acc['C'],
}

routing_list = list(CLASS_ROUTING.items())  # same order as class_indices

fig, ax = plt.subplots(figsize=(14, 5))

gains, bar_colors_v, labels_v = [], [], []
for i, (cls_full, domain) in enumerate(routing_list):
    cls_short = cls_full.replace('Tomato_','').replace('_',' ')[:22]
    # Find the index of this class in test_generator.class_indices
    if cls_full not in test_generator.class_indices:
        continue
    idx = test_generator.class_indices[cls_full]
    baseline = domain_baseline[domain][idx]
    gain = e_acc[idx] - baseline
    gains.append(gain)
    bar_colors_v.append('#4CAF50' if gain >= 0 else '#F44336')
    labels_v.append(f'{cls_short}\n({domain})')

bars = ax.bar(range(len(gains)), gains, color=bar_colors_v,
              edgecolor='white', linewidth=0.8)

# Value labels
for j, (bar, g) in enumerate(zip(bars, gains)):
    va = 'bottom' if g >= 0 else 'top'
    offset = 0.3 if g >= 0 else -0.3
    ax.text(bar.get_x() + bar.get_width()/2, g + offset,
            f'{g:+.1f}%', ha='center', va=va, fontsize=8, fontweight='bold')

ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xticks(range(len(labels_v)))
ax.set_xticklabels(labels_v, fontsize=8, rotation=20, ha='right')
ax.set_ylabel('E accuracy − assigned domain baseline (%)')
ax.set_title('Strategy E — Routing validation: did each class beat its assigned domain?',
             fontsize=12, fontweight='bold')
good_patch = mpatches.Patch(color='#4CAF50', label='Routing validated (E >= domain baseline)')
bad_patch  = mpatches.Patch(color='#F44336', label='Routing underperformed domain baseline')
ax.legend(handles=[good_patch, bad_patch], fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, f'routing_validation_{STRATEGY}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

validated = sum(1 for g in gains if g >= 0)
print(f'\n✅ Routing validation chart saved!')
print(f'   Routing validated for : {validated}/{len(gains)} classes')
print(f'   Underperformed        : {len(gains)-validated}/{len(gains)} classes')

In [ ]:
# ============================================================
# Cell 15 — Save Model & Log Results
# ============================================================

# Save final model
final_path = os.path.join(MODEL_PATH, f'model_{STRATEGY}_final.h5')
model.save(final_path)

# Append to shared results summary
results_path = os.path.join(OUTPUT_PATH, 'results_summary.txt')
with open(results_path, 'a') as f:
    f.write(f'\nStrategy {STRATEGY}\n')
    f.write(f'  Routing         : 4 color / 3 segmented / 3 mixed\n')
    f.write(f'  Test Accuracy   : {test_accuracy:.4f}\n')
    f.write(f'  Test Loss       : {test_loss:.4f}\n')
    f.write(f'  Best Val Acc    : {max(history.history["val_accuracy"]):.4f}\n')
    f.write(f'  vs Strategy A   : {(test_accuracy - 0.8906)*100:+.2f}%\n')
    f.write(f'  vs Strategy D   : {(test_accuracy - 0.8842)*100:+.2f}%\n')
    f.write(f'  E class wins    : {e_wins}/10\n')

print(f'✅ Final model saved   : {final_path}')
print(f'📝 Results logged to  : {results_path}')

print(f'\n📁 Models folder now contains:')
for fname in sorted(os.listdir(MODEL_PATH)):
    size = os.path.getsize(os.path.join(MODEL_PATH, fname)) / (1024*1024)
    print(f'   {fname} — {size:.1f} MB')

print(f'\n📁 Outputs saved:')
e_files = [f for f in os.listdir(OUTPUT_PATH) if STRATEGY in f or 'ABCDE' in f]
for fname in sorted(e_files):
    print(f'   {fname}')

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Strategy E complete!
  Next steps:
    → 06_gradcam.ipynb        (all 5 models)
    → 07_severity_estimator.ipynb
    → 08_robustness_testing.ipynb
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")